# 01 — Dataset Audit and Integrity Verification

**Project:** Vegetable Quality and Freshness Detection Using Image Processing and Deep Learning

This notebook verifies the dataset before any modelling. It reads the artefacts
produced by `src/data/audit.py` — it does not re-invent them — so the numbers
here are the same numbers the tests assert on.

Audit answers four questions:

1. Does every file actually decode?
2. What are the labels, and are they usable?
3. Are there duplicates that would leak across a train/test boundary?
4. What are the image properties the pipeline must handle?


## 1. Provenance

In [1]:
import sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)

# Locate the project root by walking upward until config.py is found.  This is
# robust to the kernel's working directory being either the project root or the
# notebooks/ directory, which differs between nbconvert, JupyterLab and VS Code.
_here = Path.cwd().resolve()
ROOT = _here
for _cand in (_here, *_here.parents):
    if (_cand / "config.py").exists():
        ROOT = _cand
        break
else:
    raise RuntimeError(f"could not locate project root from {_here}")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import config

GRAPHS = "evaluation/graphs"
from IPython.display import Image, display, Markdown
from src.utils import load_json, fmt_bytes, imread_unicode

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("kernel cwd :", _here)
print("project root:", ROOT)
print("python      :", sys.version.split()[0])


kernel cwd : C:\Users\Sachin\OneDrive\Documents\Skanda's IPCV proj\Notebooks
project root: C:\Users\Sachin\OneDrive\Documents\Skanda's IPCV proj
python      : 3.12.16


In [2]:
summary = load_json(config.METADATA_DIR / "audit_summary.json")
source = summary["source"]
display(pd.DataFrame(
    {"field": list(source), "value": [str(v) for v in source.values()]}
))
print("audit wall time:", summary["audit_duration_s"] , "s")

,field,value
0,mendeley,https://data.mendeley.com/datasets/6nxnjbn9w6/1
1,doi,10.17632/6nxnjbn9w6.1
2,paper_doi,10.1016/j.dib.2022.108657
3,huggingface,https://huggingface.co/datasets/Project-AgML/V...
4,licence,CC BY 4.0
5,upstream_reported_images,6850
6,local_variant,"Hugging Face redistribution, 6150 images, resi..."


audit wall time: 17.65 s


The local corpus is the **Hugging Face redistribution**: 6,150 images already
resized to 256×256. The Mendeley original and the *Data in Brief* paper both
describe **6,850** images, so 700 are absent here. That gap is recorded as a
dataset limitation (see `Admin/decisions.md`, D-004) rather than silently ignored,
and it is the reason no benchmark numbers are claimed against published results.


## 2. Integrity

In [3]:
inventory = pd.read_csv(config.METADATA_DIR / "image_inventory.csv")
counts, storage, geom = summary["counts"], summary["storage"], summary["geometry"]

integrity = pd.DataFrame({
    "check": ["files enumerated", "decoded successfully", "failed to decode",
              "total size", "mean file size", "distinct dimensions",
              "colour channels", "Pillow modes", "extensions"],
    "value": [
        counts["total_files"], counts["readable"], counts["unreadable"],
        fmt_bytes(storage["total_bytes"]),
        f"{storage['mean_bytes_per_image']:.0f} B",
        geom["unique_dimensions"], geom["channels"], geom["pillow_modes"],
        geom["file_extensions"],
    ],
})
display(integrity)

assert counts["unreadable"] == 0, "corrupt files present"
assert counts["total_files"] == 6150
print("PASS - all files decode; corpus is 6,150 images /", fmt_bytes(storage["total_bytes"]))

,check,value
0,files enumerated,6150
1,decoded successfully,6150
2,failed to decode,0
3,total size,125.5 MiB
4,mean file size,21405 B
5,distinct dimensions,{'256x256': 6150}
6,colour channels,{'3': 6150}
7,Pillow modes,{'RGB': 6150}
8,extensions,{'jpg': 6150}


PASS - all files decode; corpus is 6,150 images / 125.5 MiB


Every file was opened twice: Pillow `verify()` (which catches truncated JPEG
data that OpenCV tolerates) and a full OpenCV decode. Zero failures means the
audit's other numbers are computed over the whole corpus.


## 3. Label structure

In [4]:
cells = pd.DataFrame(counts["cells"])
grid = cells.pivot(index="vegetable", columns="quality", values="count")
grid = grid.reindex(index=list(config.VEGETABLE_CLASSES), columns=list(config.QUALITY_CLASSES))
display(grid.fillna(0).astype(int))

flat = cells["count"].to_numpy()
nonempty = flat[flat > 0]
print("vegetables:", len(config.VEGETABLE_CLASSES), " quality classes:", len(config.QUALITY_CLASSES))
print("populated cells:", counts["populated_cells"], "of 20")
print("empty cells:", counts["empty_cells"])
print(f"imbalance, largest/smallest non-empty cell: {nonempty.max()/nonempty.min():.1f}x")
print(f"largest: {counts['largest_cell']}")
print(f"smallest non-empty: {nonempty.min()} images")

quality,Damaged,Dried,Old,Ripe,Unripe
vegetable,,,,,
Bell Pepper,31,296,349,448,52
Chile Pepper,121,593,200,183,189
New Mexico Green Chile,138,500,261,201,227
Tomato,27,0,1234,955,145


vegetables: 4  quality classes: 5
populated cells: 19 of 20
empty cells: ['Tomato/Dried']
imbalance, largest/smallest non-empty cell: 45.7x
largest: {'vegetable': 'Tomato', 'quality': 'Old', 'count': 1234}
smallest non-empty: 27 images


Two structural facts drive every later decision:

* **`Tomato/Dried` contains zero images.** The directory exists but is empty, so
  only 19 of the 20 (vegetable, quality) cells are populated. A 20-way joint
  classifier is therefore impossible without inventing data.
* The corpus is **imbalanced by up to 45×** between the largest cell
  (`Tomato/Old`, 1,234) and the smallest (`Tomato/Damaged`, 27). Accuracy alone
  would be a misleading metric, which is why macro F1 is the headline metric
  throughout this project.

Both facts are pinned by tests in `tests/test_data.py` so they cannot drift
silently.


## 4. Duplicates and leakage risk

In [5]:
dups = load_json(config.METADATA_DIR / "duplicate_groups.json")
d = summary["duplicates"]
display(pd.DataFrame({
    "kind": ["byte-identical (SHA-256)", "near-duplicate (dHash)", "groups spanning different labels"],
    "groups": [d["exact_group_count"], d["near_group_count"], d["label_conflicting_group_count"]],
    "images": [d["exact_image_count"], d["near_image_count"], d["label_conflicting_image_count"]],
}))

top = pd.DataFrame([{
    "size": g["size"],
    "max_hamming": g["max_hamming_within"],
    "labels": " | ".join(f"{a}/{b}" for a, b in g["labels"]),
    "example": g["members"][0],
} for g in dups["near_duplicate_groups"][:8]])
display(top)

,kind,groups,images
0,byte-identical (SHA-256),0,0
1,near-duplicate (dHash),572,2182
2,groups spanning different labels,42,866


,size,max_hamming,labels,example
0,732,39,Bell Pepper/Damaged | Bell Pepper/Dried | Bell...,1. Bell Pepper/Damaged/IMG_20220609_091124.jpg
1,9,16,Tomato/Old,4. Tomato/Old/IMG_20220530_084033.jpg
2,9,14,Tomato/Old,4. Tomato/Old/IMG_20220530_084255.jpg
3,8,17,Chile Pepper/Damaged | Tomato/Ripe,2. Chile Pepper/Damaged/IMG_20220916_080708.jpg
4,7,12,Bell Pepper/Old | Tomato/Old,1. Bell Pepper/Old/IMG_20220607_091039.jpg
5,7,14,Bell Pepper/Ripe,1. Bell Pepper/Ripe/IMG_20220531_092306.jpg
6,7,12,Chile Pepper/Old,2. Chile Pepper/Old/IMG_20220604_094319.jpg
7,7,10,Chile Pepper/Unripe | Tomato/Old,2. Chile Pepper/Unripe/IMG_20220531_093432.jpg


There are **no byte-identical duplicates**, but there are many perceptual
near-duplicates: bursts of photographs of the same physical vegetable taken
seconds apart. This is the single biggest threat to a valid evaluation.

If such a burst is split randomly, the same object appears in both train and
test and the test score measures memorisation. `src/data/splits.py` therefore
treats every near-duplicate cluster as **indivisible**.

The label-conflicting clusters are a separate finding: the same-looking image
carries two different quality labels. Some are genuinely borderline (an "Old"
vegetable that is also "Dried"); others are label noise. They are kept — deleting
them would silently reshape the dataset — but they set a ceiling on achievable
accuracy and are discussed in `reports/limitations.md`.


## 5. Capture-condition variation

In [6]:
stats = pd.DataFrame(summary["image_statistics"]).T
display(stats[["mean", "std", "p05", "median", "p95"]].round(2))

print("images with sharpness below the 5th percentile:",
      int((inventory["sharpness"] < inventory["sharpness"].quantile(0.05)).sum()))
print("images with >1% clipped highlights:",
      int((inventory["clipped_white_frac"] > 0.01).sum()))
print("images with >1% clipped shadows:",
      int((inventory["clipped_black_frac"] > 0.01).sum()))

,mean,std,p05,median,p95
brightness,178.51,12.47,156.68,179.24,197.33
contrast,41.00,9.19,25.70,41.80,55.11
sharpness,139.33,132.10,12.95,92.21,431.80
saturation,56.38,26.03,18.56,51.76,100.69
colourfulness,50.89,22.47,17.37,49.09,89.16
clipped_black_frac,0.00,0.00,0.00,0.00,0.00
clipped_white_frac,0.00,0.00,0.00,0.00,0.00


images with sharpness below the 5th percentile: 308
images with >1% clipped highlights: 0
images with >1% clipped shadows: 0


Brightness is tightly controlled (mean ≈179 of 255), consistent with studio
photography on white backgrounds. Sharpness varies by two orders of magnitude,
so motion blur and focus are genuine sources of variation — a model that keys on
high-frequency detail will be unstable on the blurred tail. This motivates
measuring a sharpness-retention cost for every denoising operation in notebook 03.


## 6. Gate 1 verdict

In [7]:
checks = {
    "all images decodable": counts["unreadable"] == 0,
    "label structure verified (19/20 cells)": counts["populated_cells"] == 19,
    "single geometry (256x256)": len(geom["unique_dimensions"]) == 1,
    "duplicate clusters identified": d["near_group_count"] > 0,
    "class labels usable (>=5 classes, >=4 vegetables)":
        inventory["quality"].nunique() == 5 and inventory["vegetable"].nunique() == 4,
}
display(pd.DataFrame({"gate check": list(checks), "passed": list(checks.values())}))
assert all(checks.values())
print("GATE 1 (dataset integrity) SATISFIED")

,gate check,passed
0,all images decodable,True
1,label structure verified (19/20 cells),True
2,single geometry (256x256),True
3,duplicate clusters identified,True
4,"class labels usable (>=5 classes, >=4 vegetables)",True


GATE 1 (dataset integrity) SATISFIED
